### 실습예제 4-2. 어떤 단어가 함께 나오나? — 동시출현 네트워크

슬라이드 "질문에 맞는 분석 기법 고르기" 표의 두 번째 행을 실습합니다.

| 알고 싶은 것 | 기법 | 입력 | 평가·확인 방법 |
|---|---|---|---|
| 어떤 단어가 함께 나오나? | 동시출현 네트워크 | 단어 쌍 빈도 | 중심성, 커뮤니티 |

**목차**

1. NSMC 리뷰 불러오기 + 정제·토큰화
2. 단어 쌍 빈도 계산 — 같은 리뷰 안에 같이 등장한 단어 쌍 세기
3. `networkx`로 동시출현 네트워크 그래프 만들기
4. 중심성(centrality) — 네트워크의 '허브' 단어 찾기
5. 커뮤니티(community) 탐지 — 함께 뭉치는 단어 그룹 찾기
6. 평가·확인: 네트워크 시각화 + 원문 확인

> **이 버전에서 고친 점(폰트 경고 `findfont: Font family 'NanumGothic' not found`)**
>
> 1. **`fm.fontManager = fm.FontManager()`가 원인**이었습니다. 이 줄은 새 폰트 관리자를
>    만들어 이름만 바꿔 끼울 뿐이고, 실제로 글자를 그릴 때 쓰는 `fm.findfont`는
>    import 시점의 **옛 관리자**(나눔고딕이 없는 캐시)를 계속 가리킵니다. 그래서
>    폰트를 설치해도 경고가 반복되고 한글이 □로 깨졌습니다.
>    → `fm.fontManager.addfont(경로)`로 **현재 관리자에 직접 등록**하도록 바꿨습니다
>    (런타임 재시작 불필요).
> 2. `nx.draw_networkx_labels()`는 기본값이 `font_family="sans-serif"`라서
>    전역 폰트 설정을 무시합니다. → `font_family=FONT_NAME`을 명시했습니다.
> 3. 폰트 설치/설정 셀이 여러 군데(설치·캐시 삭제·koreanize·중복 import)에 흩어져
>    서로 덮어쓰던 것을 **1번 셀(설치) + 폰트 설정 셀 하나**로 정리했습니다.
>    nanum 폰트 설치가 안 되는 환경(Windows 등)에서는 자동으로
>    `koreanize-matplotlib`으로 대체합니다.


In [ ]:
# ① 필요한 패키지 + 한글 폰트 설치 (Colab 런타임마다 한 번)
!pip install -q kiwipiepy networkx matplotlib koreanize-matplotlib
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

#### 0. 한글 폰트 설정 (그래프를 그리기 전에 한 번만 실행)

`addfont()`로 현재 실행 중인 matplotlib에 나눔고딕을 바로 등록하므로 **런타임 재시작이 필요 없습니다.**
아래 셀 이후에는 다른 곳에서 `font.family`를 다시 바꾸지 마세요.

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

FONT_PATH = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

if os.path.exists(FONT_PATH):
    # 현재 폰트 관리자에 직접 등록 — fm.fontManager를 새로 만들어 바꿔 끼우면 안 된다
    # (글자를 그릴 때 쓰는 fm.findfont는 기존 관리자를 계속 쓰기 때문)
    fm.fontManager.addfont(FONT_PATH)
    FONT_NAME = fm.FontProperties(fname=FONT_PATH).get_name()   # 'NanumGothic'
else:
    # 나눔 폰트를 설치할 수 없는 환경(Windows/Mac 로컬 등) — 폰트가 내장된 패키지로 대체
    import koreanize_matplotlib  # noqa: F401  (import만 해도 폰트 등록 + 설정됨)
    FONT_NAME = plt.rcParams["font.family"][0]

plt.rcParams["font.family"] = FONT_NAME
plt.rcParams["axes.unicode_minus"] = False   # 마이너스(-) 기호가 □로 깨지는 것 방지
print("한글 폰트:", FONT_NAME)

#### 1. NSMC 리뷰 불러오기 + 정제·토큰화

실습예제 1-2 → 2-2 → 2-5 파이프라인을 그대로 이어 붙입니다.

In [ ]:
import re
import pandas as pd
from kiwipiepy import Kiwi

# ① NSMC 리뷰 불러오기 + 결측치·중복 제거 + 긍정·부정 균형 표본 추출 (ex1-2와 동일)
BASE = "https://raw.githubusercontent.com/e9t/nsmc/master/"
raw = pd.read_csv(BASE + "ratings_train.txt", sep="\t")
raw = raw.dropna(subset=["document"]).drop_duplicates(subset=["document"])
df = (raw.groupby("label").sample(4000, random_state=42)
         .reset_index(drop=True).rename(columns={"document": "text"}))

# ② 전각→반각 변환표 (ex2-2에서 확인한 NFKC 자모 삭제 버그 회피용)
#    0x3000(전각 공백)은 예외로 따로 매핑하고, 나머지 전각 영숫자·기호
#    (U+FF01~U+FF5E)는 반각과의 코드포인트 차이(0xFEE0)를 이용해 일괄 계산
FULLWIDTH_MAP = {0x3000: 0x20}
FULLWIDTH_MAP.update({cp: cp - 0xFEE0 for cp in range(0xFF01, 0xFF5F)})

# ③ 정제 함수 (ex2-2와 동일)
def clean(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)                      # HTML 태그 제거
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)        # URL 제거
    text = re.sub(r"([ㅋㅎㅠㅜ])\1{2,}", r"\1\1", text)        # ㅋㅋㅋㅋ → ㅋㅋ로 축소
    text = re.sub(r"(.)\1{3,}", r"\1\1\1", text)               # 그 외 반복 문자 축소
    text = text.translate(FULLWIDTH_MAP)                       # 전각 → 반각
    text = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]", " ", text)  # 한글·영숫자 외 기호 제거
    return re.sub(r"\s+", " ", text).strip().lower()           # 공백 정리 + 소문자화

df["clean"] = df["text"].apply(clean)

# ④ Kiwi 형태소 분석기 + 사전 준비 (ex2-5와 동일)
kiwi = Kiwi()
kiwi.add_user_word("꿀잼", "NNG")                 # 신조어 "꿀잼"이 "꿀"+"잼"으로 쪼개지지 않도록 등록
NOUN, PRED = {"NNG", "NNP"}, {"VV", "VA"}         # 남길 품사: 명사(NNG,NNP) + 용언(VV,VA)
STOPWORDS = {"것", "수", "하다", "나오다", "있다", "정도", "영화", "평점"}  # 불용어
SYNONYMS  = {"재미있다": "재밌다", "꿀잼": "재밌다"}                         # 동의어 통일

# ⑤ 토큰화 함수 (ex2-5와 동일): 띄어쓰기 교정 → 형태소 분석 → 품사 선택 →
#    동의어 통일 → 불용어 제거
def tokenize(text):
    out = []
    for t in kiwi.tokenize(kiwi.space(text)):   # kiwi.space(): 띄어쓰기 자동 교정
        if t.tag in NOUN and len(t.form) > 1:   # 2글자 이상 명사만
            w = t.form
        elif t.tag in PRED:                      # 용언은 기본형('~다')으로 복원
            w = t.form + "다"
        else:
            continue
        w = SYNONYMS.get(w, w)                   # 동의어 치환
        if w not in STOPWORDS:                   # 불용어 제거
            out.append(w)
    return out

df["tokens"] = df["clean"].apply(tokenize)

# ⑥ 빈 토큰 리뷰 제외 — 정제·토큰화 후 아무 단어도 안 남은 리뷰는 분석 대상에서 뺀다
df = df[df["tokens"].map(len) > 0].reset_index(drop=True)
print(f"전처리 완료: {len(df):,}건")

#### 2. 단어 쌍 빈도 계산

'동시출현(co-occurrence)'이란, 같은 문서(여기서는 같은 리뷰) 안에 두 단어가
함께 등장하는 것을 말합니다. 리뷰 하나 안의 단어들을 중복 없이 뽑아,
모든 단어 쌍(2개 조합)의 등장 횟수를 셉니다.

In [ ]:
from collections import Counter
from itertools import combinations

# ① 너무 드문 단어는 우연한 동시출현이 많으므로, 전체 10회 이상 나온 단어만 사용
#    (예: 리뷰 전체에 딱 1번 나온 단어 둘이 우연히 같은 리뷰에 있었다고 해서
#    의미 있는 연관성이라고 보기 어렵다 — 최소 등장 횟수로 '노이즈'를 거른다)
word_freq = Counter(t for toks in df["tokens"] for t in toks)
vocab = {w for w, c in word_freq.items() if c >= 10}

# ② 리뷰 한 건 안의 단어들을 모든 쌍(2개 조합)으로 나눠 등장 횟수를 센다
pair_counts = Counter()
for toks in df["tokens"]:
    # set()으로 중복 제거 — 한 리뷰에 같은 단어가 여러 번 나와도 '쌍'은 한 번만 센다
    # sorted()로 정렬 — (a,b)와 (b,a)를 같은 쌍으로 취급하기 위한 정규화
    uniq = sorted(set(t for t in toks if t in vocab))
    for a, b in combinations(uniq, 2):   # 가능한 모든 2개 조합을 생성
        pair_counts[(a, b)] += 1

# ③ 가장 자주 같이 나온 단어 쌍 확인
print("가장 자주 같이 나온 단어 쌍 top 10:")
for (a, b), c in pair_counts.most_common(10):
    print(f"  ({a}, {b})  {c}회")

#### 3. `networkx`로 동시출현 네트워크 그래프 만들기

단어 쌍 빈도를 그래프(노드=단어, 엣지=함께 등장한 관계)로 바꿉니다. 너무 적게
같이 나온 쌍(3회 미만)은 노이즈로 보고 엣지에서 제외합니다.

In [ ]:
import networkx as nx

# ① 그래프 생성: 노드=단어, 엣지=함께 등장한 관계
#    너무 적게(3회 미만) 같이 나온 쌍은 노이즈로 보고 엣지에서 제외
G = nx.Graph()
for (a, b), w in pair_counts.items():
    if w >= 3:
        G.add_edge(a, b, weight=w)   # weight: 두 단어가 함께 등장한 횟수(엣지의 '굵기' 정보)

print(f"노드(단어) 수: {G.number_of_nodes()}")
print(f"엣지(동시출현 관계) 수: {G.number_of_edges()}")

#### 4. 중심성(centrality) — 네트워크의 '허브' 단어 찾기

'연결 중심성(degree centrality)'은 한 단어가 몇 개의 다른 단어와 연결돼 있는지를
0~1 사이 값으로 나타냅니다. 값이 높을수록 리뷰 전반에 걸쳐 다양한 단어와
함께 쓰이는 '허브' 단어라는 뜻입니다.

In [ ]:
# ① 연결 중심성(degree centrality): 한 단어가 몇 개의 다른 단어와 연결돼
#    있는지를, 전체 노드 수로 나눠 0~1 사이 값으로 나타낸다
#    — 값이 높을수록 네트워크 안에서 폭넓게 함께 쓰이는 '허브' 단어
deg_cent = nx.degree_centrality(G)
top_deg = sorted(deg_cent.items(), key=lambda x: -x[1])[:10]   # 중심성 높은 순 정렬 후 상위 10개

print("연결 중심성 상위 10 (네트워크의 허브 단어):")
for w, c in top_deg:
    print(f"  {w:<8} {c:.3f}")

print("\n→ '보다'·'없다'·'좋다'처럼 영화 리뷰 어디에나 등장할 법한 일반적인")
print("  단어들이 허브로 나온다 — 리뷰라는 도메인 특성상 자연스러운 결과다.")

#### 5. 커뮤니티(community) 탐지 — 함께 뭉치는 단어 그룹 찾기

'커뮤니티'는 네트워크 안에서 서로 긴밀하게 연결된 단어들의 하위 그룹입니다.
`greedy_modularity_communities`는 전체 그래프를 모듈성(그룹 내부 연결은 촘촘하고
그룹 간 연결은 성긴 정도)이 가장 높아지는 방향으로 자동 분할합니다.

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

# ① 모듈성(그룹 내부 연결은 촘촘하고 그룹 간 연결은 성긴 정도)이 가장
#    높아지는 방향으로 전체 그래프를 자동 분할 — 이렇게 나뉜 단어 그룹이 '커뮤니티'
#    weight="weight": 동시출현 횟수가 많을수록 '더 강하게 연결된' 것으로 반영
communities = greedy_modularity_communities(G, weight="weight")
print(f"탐지된 커뮤니티 수: {len(communities)}\n")

# ② 너무 작은(5개 미만) 커뮤니티는 의미 있는 주제 묶음으로 보기 어려우므로
#    생략하고, 그 외 커뮤니티는 예시 단어 10개만 출력
for i, c in enumerate(communities):
    if len(c) < 5:
        continue
    print(f"커뮤니티 {i} (단어 {len(c)}개) 예시:", list(c)[:10])

#### 6. 평가·확인: 네트워크 시각화 + 원문 확인

숫자(중심성)와 그룹(커뮤니티)만으로는 와닿지 않으므로, 상위 허브 단어를 중심으로
네트워크를 직접 그려보고, 대표적인 단어 쌍이 실제 어떤 리뷰에서 나왔는지
원문도 확인합니다. (한글 폰트는 0번 셀에서 이미 설정했습니다.)

In [ ]:
# ① 중심성 상위 25개 단어만 뽑아 부분 그래프로 시각화 (전체를 그리면 너무 복잡함)
top_words = [w for w, _ in sorted(deg_cent.items(), key=lambda x: -x[1])[:25]]
H = G.subgraph(top_words)   # 전체 그래프 G에서 이 25개 단어만 떼어낸 부분 그래프

# ② 노드 크기는 중심성에 비례하게, spring_layout(인력·척력 시뮬레이션)으로 배치
plt.figure(figsize=(9, 7))
pos = nx.spring_layout(H, seed=42, k=0.6)   # seed 고정 → 매번 같은 배치가 나와 재현 가능
sizes = [3000 * deg_cent[n] + 200 for n in H.nodes()]   # 중심성이 클수록 원을 크게
nx.draw_networkx_nodes(H, pos, node_size=sizes, node_color="#9ecae1")
nx.draw_networkx_edges(H, pos, alpha=0.3)               # 엣지는 흐리게(alpha) 그려 노드를 돋보이게
# 주의: draw_networkx_labels의 기본값은 font_family="sans-serif"라 전역 설정을 무시한다
#       → 한글 폰트를 반드시 직접 지정해야 단어가 □로 깨지지 않는다
nx.draw_networkx_labels(H, pos, font_size=11, font_family=FONT_NAME)
plt.title("동시출현 네트워크 (중심성 상위 25개 단어)")
plt.axis("off")       # 좌표축 숨기기 — 네트워크 그래프에서는 좌표 값 자체가 의미 없음
plt.tight_layout()
plt.savefig("cooccurrence_network.png", dpi=120)
plt.show()

# ③ 평가·확인: 가장 자주 같이 나온 단어 쌍 1위가 실제로 어떤 리뷰에서 나왔는지 확인
a, b = pair_counts.most_common(1)[0][0]
examples = df[df["tokens"].apply(lambda t: a in t and b in t)]["text"].head(3)
print(f"\n'{a}'와 '{b}'가 함께 나온 리뷰 예시:")
for ex in examples:
    print("  -", ex[:60])

**정리**: 동시출현 네트워크는 '어떤 단어가 함께 쓰이는가'라는 관계 자체를
분석 대상으로 삼습니다. 중심성은 전체 리뷰를 관통하는 '허브' 단어를, 커뮤니티는
서로 긴밀하게 묶이는 '주제 집단'을 보여줍니다. 다만 숫자와 그래프만으로는
해석이 틀리기 쉬우므로, 반드시 원문 리뷰로 확인해야 합니다.